# Let's give our model tools to work with

In [1]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # automatically load .env file if it exists

True

In [4]:
import anthropic

client = anthropic.Anthropic()

model = "claude-haiku-4-5"

## Define tool

### Functions

In [5]:
def get_current_time():
    
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

### Schema Tools

In [6]:
tools = [
    {
        "name": "get_current_time",
        "description": "Get the current date and time in the format YYYY-MM-DD HH:MM:SS",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    }
]

TOOL_FUNCS = {"get_current_time": get_current_time}

### LOOP the agent

In [ ]:
messages = []

while True:
    
    user_input = input("User: ")
    
    if user_input.lower() in ["exit", "quit", "q"]:
        print("Exiting the chat.")
        break
    
    messages.append({"role": "user", "content": user_input})
    
    while True:
        response = client.messages.create(
            model=model, messages=messages, tools=tools, max_tokens=1048,
        )
        # (Bug 3 fix) always record the assistant's FULL reply — text or tool_use
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            break   # final answer -> leave the inner loop

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = TOOL_FUNCS[block.name](**block.input)
                tool_results.append({
                    "type": "tool_result",        # (Bug 2 fix) "type", not "tool"
                    "tool_use_id": block.id,
                    "content": str(result),
                })
        messages.append({"role": "user", "content": tool_results})

    # (Bug 1 fix) OUTSIDE the inner loop — only the final text reaches here
    answer = "".join(b.text for b in response.content if b.type == "text")
    print(f"Agent: {answer}")

User:  What time is it?


Agent: The current time is **July 2, 2026 at 11:27:21 PM** (23:27:21 in 24-hour format).


User:  1 + 1


Agent: 1 + 1 = **2**


User:  * 3


Agent: 2 * 3 = **6**


User:  *7


Agent: 6 * 7 = **42**
